# D5.3 · Validating the fix against the KCIs

**Function D — The Agentic SOC → The Agentic SOC — Recover and Root Cause**  ·  *Security of AI*

Builds on **[D5.2 · The root cause record](https://spbreed.github.io/cyber-commons/lessons/D5.2.html)**.

| | |
|---|---|
| Tools used | OPA |

## What this lesson is

**What it covers.** Re-measuring the key control indicators an incident moved, after the fix, and reporting which were actually restored.

**Why a security engineer needs it.** A closed ticket is not evidence that a control came back. Automating the re-measurement is what makes it happen on every fix rather than the memorable ones — and it routinely finds that an indicator improved without reaching target, which reads as "done" in every system that does not check.

## 1 · The hook

The ticket is closed and the incident is marked remediated. Re-measure the indicators it moved and two of them never came back — including the detection time, which improved from 194 minutes to 118 and is still eight times its target.

> **At CyberTravels.** The six indicators are CyberTravels' own, and two of them do not come back: refunds carrying an approval, and the time to detect a scope breach. CyberTravels' detection went from 194 minutes to 118 against a fifteen-minute target — a real improvement that would still let the same incident run for two hours.

## 2 · The framework

```
   kci        healthy   during     after fix    verdict

   KCI-01      1.00      0.41       1.00        restored
   KCI-02      1.00      0.00       1.00        restored
   KCI-03      1.00      0.86       0.94        NOT restored
   KCI-04      9 min     194 min    118 min     NOT restored
   KCI-05      1.00      0.97       1.00        restored

                                    ^
                                    |
              the incident record would have said "remediated"
              for all five. improvement is not restoration.
```

A fix is a claim. The key control indicators built in E1.1 are how the claim
gets checked: re-measure the indicators the incident moved, and see which came
back to target.

Doing it automatically is the point. Done by hand, it happens for the incidents
somebody remembers — which are not the ones where it matters.

## 3 · Improvement is not restoration

The uncomfortable row in this run is detection time. It improved from 194
minutes to 118 and is still eight times its target, so the control the root
cause record named has been *partially* built.

Partially built is not built. Without the re-measurement it reads as done, and
the incident record would have said "remediated" for every indicator the
incident touched.

## 4 · Six indicators, three readings each

Healthy, during the incident, and after the fix. Restored and not-restored are reported separately, and untouched indicators are kept out of the report entirely.

### The skill — [`skills/response/kci-fix-validation/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/kci-fix-validation/SKILL.md)

```yaml
name: kci-fix-validation
description: >-
  Re-measure the key control indicators an incident moved and report which the
  fix actually restored. Use when closing a remediation, when a ticket is marked
  done without evidence, or when a control was rebuilt and nobody has checked
  whether it now meets its target.
allowed-tools: Read, Grep, Glob
```

# Closing a ticket is not evidence that a control came back

A fix is a claim. The KCIs are how the claim is checked: re-measure the
indicators the incident moved and see which returned to target.

Doing it automatically is the point. Done by hand, it happens for the incidents
somebody remembers, which are not the ones where it matters.

## When to use this

On every remediation that claims to restore a control, before the incident is
closed — and again at the next measurement cycle, because a fix that holds for a
week and then regresses is common.

## Step-by-step

**1 — Take the KCIs from the root cause record.** Not all of them; the ones the
incident actually moved.

**2 — Read the healthy baseline, not just the target.** A control that was
never at target before the incident did not regress.

**3 — Re-measure after the fix, the same way.** A different measurement is a
different indicator.

**4 — Report restored, not restored, and untouched separately.** Untouched
indicators are noise in the report and belong out of it.

**5 — Attach before/during/after to the incident record.** That is the artefact
an auditor asks for, and it is worth more than the postmortem.

## Example

**Input** — six indicators with three readings each, in
[`scripts/kci_fix_validation.py`](scripts/kci_fix_validation.py).

**Output** — the part that changes the outcome:

```
   KCI-03  refunds with a matching approval
          target >= 0.99, measured 0.94 — the fix did not reach this one
   KCI-04  mean minutes to detect a scope breach
          target <= 15, measured 118.0 — the fix did not reach this one
```

Detection improved from 194 minutes to 118 and is still eight times target.
Without the re-measurement, that reads as done.

## Output contract

```json
{
  "restored": ["str"],
  "not_restored": ["str"],
  "regressed": ["str"]
}
```

## Common edge cases

- **Partially restored.** Improvement is not restoration; the target is the
  test, not the direction of travel.
- **The indicator was already failing before the incident.** Then it is a
  standing gap, not incident damage.
- **The measurement changed with the fix.** A new instrument reading better is
  not a control reading better.

## Failure modes

- **Marking remediated on ticket closure.** The most common way a control gap
  survives its own incident.
- **Reporting an average across indicators.** Two restored and two not is not
  "50% recovered", it is two open gaps.
- **Measuring once.** Regression at the next cycle is the normal case.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/kci-fix-validation/scripts/kci_fix_validation.py
SCRIPT = "skills/response/kci-fix-validation/scripts/kci_fix_validation.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.2 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Three indicators restored and two not, against an incident record that would have called all five remediated.

## Your turn

Add a sixth indicator that regressed — better than target before, worse after. The report has nowhere to put it yet.

---

**Next → [D5.4 · Post-incident change surface](https://spbreed.github.io/cyber-commons/lessons/D5.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D5.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D5.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*